# MitoFish Dataset Preprocessing for Phylogenetic Embedding

This notebook prepares a curated mitochondrial gene dataset for training a transformer model that learns gene-invariant species embeddings.

The preprocessing pipeline integrates:

- MitoFish mitochondrial gene sequences
- MitoFish taxonomy metadata
- A reference fish phylogenetic tree (The Fish Tree of Life)

The goal is to construct a dataset where each sequence is linked to a **species present in the phylogenetic tree**, allowing later alignment between sequence embeddings and evolutionary relationships.

Main preprocessing steps:

1. Load and type-optimize MitoFish annotation tables
2. Join sequence annotations with taxonomy metadata
3. Parse mitochondrial sequences from FASTA
4. Merge sequences with gene and taxonomy annotations
5. Extract species names from the phylogenetic tree
6. Resolve species name mismatches between the tree and MitoFish
7. Filter sequences to species present in the tree
8. Generate dataset statistics
9. Export the final processed dataset

## Import libraries and define dataset paths

This section imports the required Python libraries and defines the file paths used throughout the preprocessing pipeline.

Key dependencies:

- **pandas** for data manipulation
- **ETE3** for parsing the phylogenetic tree
- **matplotlib** for quick exploratory plots

Dataset directories:

- `datasets/` contains the raw MitoFish tables, FASTA sequences and phylogenetic tree
- `output/` will contain the processed dataset

In [40]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from ete3 import Tree

path_dataset = Path("datasets")
path_out_dir = Path("output")

path_fishtree = path_dataset / "fishtree"
path_fishtree_taxonomy = path_fishtree / "PFC_taxonomy.csv"
path_fishtree_tree = path_fishtree / "actinopt_12k_raxml.tre"

path_mitofish = path_dataset / "MitoFish"
path_mitofish_fasta = path_mitofish / "mitofishdb.fa"
path_mitofish_seq_annotation = path_mitofish / "seq_annotation.parquet"
path_mitofish_seq_description = path_mitofish / "seq_description.parquet"
path_mitofish_seq_heterospecific = path_mitofish / "seq_heterospecific.parquet"
path_mitofish_seq_taxonid = path_mitofish / "seq_taxonid.parquet"
path_mitofish_taxonid_name = path_mitofish / "taxonid_name.parquet"
path_mitofish_taxonid_speciesid = path_mitofish / "taxonid_speciesid.parquet"
path_mitofish_speciesid_lineageid = path_mitofish / "speciesid_lineageid.parquet"

## Load MitoFish sequence annotations

MitoFish provides detailed annotation tables describing the mitochondrial genes present in each sequence record.

This table contains:

- accession identifiers
- gene name
- start and end positions within the mitochondrial genome
- strand orientation
- sequence length

To reduce memory usage, several columns are cast to more efficient data types:

- `gene` → categorical
- `start`, `end`, `length` → 32-bit integers
- `accession` → string

In [41]:
df_mf_seq_an = pd.read_parquet(path_mitofish_seq_annotation).astype({
    'accession': 'string',
    'gene': 'category',
    'start': 'int32',
    'end': 'int32',
    'strand': 'category',
    'length': 'int32'})
df_mf_seq_an

,accession,gene,start,end,strand,length
0,AB000667,12S_rRNA,2469,3417,1,949
1,AB000667,16S_rRNA,3491,3576,1,86
2,AB000667,Cytb,2,857,1,856
3,AB000673,ND5,194,1008,1,815
4,AB000674,COXIII,1,233,1,233
...,...,...,...,...,...,...
1004508,Z97553,Cytb,1,363,1,363
1004509,Z97554,Cytb,1,363,1,363
1004510,Z97555,Cytb,1,363,1,363
1004511,Z97556,Cytb,1,363,1,363


## Load sequence–taxon mapping

Each sequence accession is associated with a **taxonomic identifier (taxon_id)**.

This table links sequence accessions to taxonomic identifiers and creation metadata.

The taxonomic identifier will later be used to retrieve:

- species identity
- taxonomic lineage

In [42]:
df_mf_seq_taxid = pd.read_parquet(path_mitofish_seq_taxonid).astype({
    'accession': 'string',
    'create_date': 'datetime64[ns]',
    'taxon_id': 'int32'
})
df_mf_seq_taxid

,accession,create_date,taxon_id
0,AB000667,1997-02-05,8255
1,AB000668,1997-02-05,8255
2,AB000669,1997-02-05,8255
3,AB000670,1997-02-05,8255
4,AB000671,1997-02-05,8255
...,...,...,...
947202,Z97553,1999-09-20,27763
947203,Z97554,1999-09-20,27763
947204,Z97555,1999-09-20,27763
947205,Z97556,1999-09-20,27763


## Load taxonomic lineage information

MitoFish provides lineage tables describing the hierarchical taxonomy associated with each taxon identifier.

This includes taxonomic ranks such as:

- order
- family
- genus
- species

These tables allow reconstruction of the full taxonomic lineage for each sequence.

In [43]:
df_mf_taxid_name = pd.read_parquet(path_mitofish_taxonid_name).astype({
    'lineage_id': 'int32',
    'lineage_rank': 'category',
    'lineage_name': 'string'
})
df_mf_taxid_name

,lineage_id,lineage_rank,lineage_name
0,1000278,subspecies,Acheilognathus tabira tohokuensis
1,1000982,species,Steindachneridion melanodermatum
2,1001000,species,Sebastes chrysomelas x Sebastes carnatus
3,100124,genus,Pseudobarbus
4,100125,species,Pseudobarbus burchelli
...,...,...,...
60912,999360,species,Hypselobarbus periyarensis
60913,999361,species,Barbodes carnaticus
60914,999678,genus,Oxygaster
60915,999679,species,Oxygaster anomalura


## Map taxon identifiers to species identifiers

MitoFish separates taxonomic identifiers (`taxon_id`) from **species identifiers (`species_id`)**.

This table links each taxon entry to the corresponding species identifier, allowing aggregation of sequences at the species level.

In [44]:
df_mf_taxid_spid = pd.read_parquet(path_mitofish_taxonid_speciesid).astype({
    'taxon_id': 'int32',
    'species_id': 'int32'
})
df_mf_taxid_spid

,taxon_id,species_id
0,1000982,1000982
1,1001000,1001000
2,100125,100125
3,100126,100126
4,1001636,1001636
...,...,...
55177,999359,999359
55178,999360,999360
55179,999361,999361
55180,999679,999679


## Retrieve full lineage hierarchy for each species

This table contains the ordered taxonomic lineage for each species identifier.

It is used to reconstruct hierarchical taxonomy information such as:

- genus
- family
- order

These fields will later be included in the final dataset.

In [45]:
df_mf_spid_linid = pd.read_parquet(path_mitofish_speciesid_lineageid).astype({
    'species_id': 'int32',
    'rank_order': 'int32',
    'rank_name': 'category',
    'lineage_id': 'int32'
})

df_mf_spid_tax = pd.merge(df_mf_spid_linid, df_mf_taxid_name, on="lineage_id")

target_ranks = ['order', 'family', 'genus', 'species']
df_mf_spid_tax = df_mf_spid_tax[df_mf_spid_tax['rank_name'].isin(target_ranks)]

df_mf_spid_tax_piv = df_mf_spid_tax.pivot(
    index='species_id',
    columns='rank_name',
    values='lineage_name'
).reset_index()

df_mf_spid_tax_piv

rank_name,species_id,species,family,genus,order
0,7748,Lampetra fluviatilis,Petromyzontidae,Lampetra,Petromyzontiformes
1,7750,Lampetra planeri,Petromyzontidae,Lampetra,Petromyzontiformes
2,7752,Lampetra aepyptera,Petromyzontidae,Lampetra,Petromyzontiformes
3,7753,Lethenteron reissneri,Petromyzontidae,Lethenteron,Petromyzontiformes
4,7755,Mordacia mordax,Petromyzontidae,Mordacia,Petromyzontiformes
...,...,...,...,...,...
54392,3683097,Ecsenius sp. T76863-1,Blenniidae,Ecsenius,Blenniiformes
54393,3683777,Elates sp. T77057-1,Platycephalidae,Elates,Perciformes
54394,3684156,Danio rerio x Danio albolineatus,Danionidae,Danio,Cypriniformes
54395,3684157,Danio rerio x Danio kyathit,Danionidae,Danio,Cypriniformes


## Merge sequence annotations with taxonomy metadata

All annotation and taxonomy tables are merged to create a unified dataset containing:

- sequence accession
- gene annotation
- taxonomic identifiers
- species identifiers
- lineage information

This step links each mitochondrial gene annotation to its corresponding species.

In [46]:
df_mf_seq_an_taxid = pd.merge(df_mf_seq_an, df_mf_seq_taxid, on="accession")
df_mf_seq_an_spid = pd.merge(df_mf_seq_an_taxid, df_mf_taxid_spid, on="taxon_id")
df_mf_seq_an_tax = pd.merge(df_mf_seq_an_spid, df_mf_spid_tax_piv, on='species_id')
df_mf_seq_an_tax.drop(columns=['create_date', 'taxon_id', 'species_id', 'start', 'end', 'strand'], inplace=True)

df_mf_seq_an_tax

,accession,gene,length,species,family,genus,order
0,AB000667,12S_rRNA,949,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
1,AB000667,16S_rRNA,86,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
2,AB000667,Cytb,856,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
3,AB000673,ND5,815,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
4,AB000674,COXIII,233,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes
...,...,...,...,...,...,...,...
1004499,Z97553,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes
1004500,Z97554,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes
1004501,Z97555,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes
1004502,Z97556,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes


## Parse mitochondrial sequences from FASTA

The mitochondrial sequences are stored in a FASTA file.

The FASTA file is parsed into a table containing:

- accession identifier
- nucleotide sequence

This allows direct merging with the annotation tables using the accession ID.

In [47]:
df_mf_fasta = pd.read_csv(path_mitofish_fasta, lineterminator='>', header=None)
df_mf_fasta = df_mf_fasta[0].str.split('\n', expand=True, n=1).set_axis(["accession", "sequence"], axis=1)
df_mf_fasta['accession'] = df_mf_fasta['accession'].str.strip()
df_mf_fasta['sequence'] = df_mf_fasta['sequence'].str.upper().str.strip()

df_mf_fasta['accession'] = df_mf_fasta['accession'].str.split(';')
df_mf_fasta = df_mf_fasta.explode('accession')

df_mf_fasta = df_mf_fasta.astype({
    'accession': 'string',
    'sequence': 'string'
})

df_mf_fasta

,accession,sequence
0,PQ550629,GGCACTGCTTTAAGCCTTCTTATTCGAGCAGAACTAAGCCAACCAG...
1,MN473720,ACTAAATCAGCCCTCGTCCATAAACCCATATCACAGGACTGAACAC...
2,GU593119,GTGATGAAACTTCGGATCCCTCCTACTACTCTGTCTCTCAGCACAA...
3,MH715312,TGGGATTAGATACCCCACTATGCTTAGTCGTAAACCTCGGTAGGCC...
4,OM978181,GGTGCCTGAGCCGGAATAGTAGGTACAGCCCTGAGCCTGCTAATTC...
...,...,...
580146,GU210318,CGCCAGACTACGAGCATAGCTTAAAACCCAAAGGACTTGGCGGTGC...
580147,MZ723175,CTAGTATTCGGTGCATGAGCTGGAATAGTTGGCACGGCCTTAAGCT...
580148,KF808193,CCTTTATTTAATCTTTGGTGCATGAGCGGGGATAGTGGGTACTGGT...
580149,KF021391,ATGCCTCAACTTAACCCCGCACCCTGGTTCCTAATTATAGTATTCT...


## Validate nucleotide alphabet

Before merging sequences with annotations, we verify the nucleotide alphabet present in the dataset.

This step ensures that sequences contain only valid nucleotide characters (e.g., A, C, G, T, N).

In [48]:
set().union(*df_mf_fasta['sequence'].unique())

{'A', 'B', 'C', 'D', 'G', 'H', 'K', 'M', 'N', 'R', 'S', 'T', 'V', 'W', 'Y'}

## Combine sequences with gene annotations

The sequence table is merged with the annotated gene dataset using the accession identifier.

Duplicate entries are removed based on the combination of:

- sequence
- gene
- species

This ensures that identical gene sequences are not counted multiple times.

In [49]:
df_mf_all_genes = pd.merge(df_mf_seq_an_tax, df_mf_fasta, on="accession")
df_mf_all_genes.drop_duplicates(subset=['sequence', 'gene', 'species'], inplace=True)

print(f"Number of species in MitoFish: {df_mf_all_genes['species'].nunique()}")
df_mf_all_genes.value_counts(subset='species').to_frame()

Number of species in MitoFish: 43808


,count
species,
Amblyraja radiata,6715
Gadus morhua,4710
Chaenogobius annularis,3163
Carcharodon carcharias,2703
Carcharhinus leucas,2594
...,...
cf. Selaroides sp. AWFS-F16-1504,1
cf. Pseudorhombus sp. AWFS-F16-1497,1
cf. Pomadasys sp. AWFS-F16-1492,1


## Load reference phylogenetic tree

The fish phylogeny is loaded using the ETE3 library.

Leaf nodes of the tree correspond to species names.

Species names are normalized by replacing underscores with spaces to match the formatting used in the MitoFish dataset.

In [50]:
t = Tree(str(path_fishtree_tree))
t_leaves = t.get_leaves()
df_leaves = pd.DataFrame([l.name.replace("_", " ") for l in t_leaves], columns=['species'], dtype='string')
df_leaves['species'] = df_leaves['species'].str.replace(r'\b(\w+) \1\b', r'\1', regex=True)

BAD_TOKENS = (" sp", " sp.", " spp", " spp.", " cf", " cf.", " aff", " aff.", " nr", " nr.")

df_leaves = df_leaves[~df_leaves['species'].str.endswith(BAD_TOKENS)]

print(f"Number of species in The Fish Tree of Life: {df_leaves['species'].nunique()}")
df_leaves

Number of species in The Fish Tree of Life: 11629


,species
0,Gambusia marshi
1,Gambusia panuco
2,Gambusia regani
3,Gambusia aurata
4,Gambusia hurtadoi
...,...
11633,Polypterus congicus
11634,Polypterus retropinnis
11635,Polypterus mokelembembe
11636,Polypterus ornatipinnis


## Identify tree species without sequence data

Not all species present in the phylogenetic tree have mitochondrial gene sequences available in MitoFish.

This step identifies tree species that are not present in the MitoFish dataset.

The missing species names are exported for manual inspection.

In [51]:
df_leaves_not_in_mf = df_leaves[~df_leaves['species'].isin(df_mf_all_genes['species'])]

path_leaves_not_in_mf =  path_fishtree / "leaves_not_in_mf"
path_leaves_not_in_mf.mkdir(exist_ok=True)

for i in range(0, df_leaves_not_in_mf.shape[0], 100):
    df_leaves_not_in_mf.iloc[i:i+100].to_csv(path_leaves_not_in_mf / f"leaves_not_in_mf_{i}-{i+100}.txt", 
                                             index=False, header=False)

print(f"Number of Fish Tree of Life species not in MitoFish: {df_leaves_not_in_mf['species'].nunique()}")
df_leaves_not_in_mf

Number of Fish Tree of Life species not in MitoFish: 1037


,species
10,Gambusia zarskei
119,Limia dominicensis
148,Micropoecilia picta
194,Cyprinodon hubbsi
198,Cyprinodon variegatus ovinus
...,...
11527,Anguilla bengalensis labiata
11532,Anguilla bicolor pacifica
11545,Anguilla australis schmidtii
11624,Polypterus polli


## Resolve species name mismatches using MitoFish mapping tool

Many species mismatches arise due to **taxonomic revisions**, such as genus changes.

To resolve these inconsistencies, the MitoFish species name mapping tool was used:

https://mitofish.aori.u-tokyo.ac.jp/species/mapping

Because the tool only accepts **100 species names per query**, the list of unmatched species was split into multiple batches.

The returned mappings were then combined into a single table.

In [52]:
path_leaves_remap = path_fishtree / "leaves_remap"

df_map = pd.concat(
    pd.read_csv(path_leaves_remap / f"table-mapping-species-names-{i}.csv") for i in range (1, 12)
)

# Only select rows with species mismatch
df_map = df_map[df_map['Need Manually Check?'] == "Yes"]
# Only select rows where Name in NCBI is not already present in Fish Tree of Life species
df_map = df_map[~df_map['Name in NCBI'].isin(df_leaves['species'])]
# Only select rows where Name in NCBI is present in MitoFish
df_map = df_map[df_map['Name in NCBI'].isin(df_mf_all_genes['species'])]
# Only select rows where the query species name matches the NCBI species name (first 3 characters)
df_map = df_map[(df_map['Query Name'].str.split().str[-1].str[:3] == df_map['Name in NCBI'].str.split().str[-1].str[:3]) |
                (df_map['Query Name'].str.split().str[-2].str[:3] == df_map['Name in NCBI'].str.split().str[-1].str[:3])]
# Drop duplicates so each query gets a single remap name
df_map.drop_duplicates('Query Name', inplace=True)

print(f"Number of Fish Tree of Life species with new names in MitoFIsh: {df_map['Query Name'].nunique()}")
df_map

Number of Fish Tree of Life species with new names in MitoFIsh: 869


,Taxon ID,Query Name,Name in NCBI,Need Manually Check?
1,154033.0,Limia dominicensis,Poecilia dominicensis,Yes
2,69234.0,Micropoecilia picta,Poecilia picta,Yes
4,2802244.0,Cyprinodon variegatus ovinus,Cyprinodon ovinus,Yes
6,30734.0,Garmanella pulchra,Jordanella pulchra,Yes
7,722643.0,Adinia xenica,Fundulus xenicus,Yes
...,...,...,...,...
36,2864443.0,Pisodonophis sangjuensis,Ophichthus sangjuensis,Yes
38,267672.0,Gnathophis nystromi ginanago,Gnathophis ginanago,Yes
39,391210.0,Poeciloconger kapala,Ariosoma kapala,Yes
41,458554.0,Bassanago albescens,Pseudoxenomystax albescens,Yes


## Apply species name remapping

The mappings returned by the MitoFish name mapping tool are applied to the tree species list.

This step corrects outdated or synonymized species names.

After remapping, the number of tree species without sequence data decreased from:

**1037 → 168 species**

In [53]:
name_mapping = dict(zip(df_map['Query Name'], df_map['Name in NCBI']))

df_leaves_remap = df_leaves.replace(name_mapping)
df_leaves_remap_not_in_mf = df_leaves[(~df_leaves_remap['species'].isin(df_mf_all_genes['species']))]

print(f"Number of Fish Tree of Life species not in MitoFish after name remapping: {df_leaves_remap_not_in_mf['species'].nunique()}")
df_leaves_remap_not_in_mf

Number of Fish Tree of Life species not in MitoFish after name remapping: 168


,species
10,Gambusia zarskei
194,Cyprinodon hubbsi
299,Aphanius farsicus
302,Aphanius splendens
436,Hypsolebias flammeus
...,...
11454,Thalassenchelys coheni
11532,Anguilla bicolor pacifica
11545,Anguilla australis schmidtii
11624,Polypterus polli


## Filter sequences to species present in the phylogenetic tree

Only sequences belonging to species present in the phylogenetic tree are retained.

This ensures that every sequence in the dataset corresponds to a species that has a known position in the phylogeny.

This alignment is required for later experiments where sequence embeddings will be compared with evolutionary distances.

## Dataset statistics

Gene-level statistics are computed to characterize the dataset, including:

- number of species represented per gene
- number of sequences per gene
- average sequence length
- number of sequence variants per species

These statistics help assess dataset balance and guide training strategies for the transformer model.

In [54]:
df_mf_all_genes_on_tree = pd.merge(df_mf_all_genes, df_leaves_remap, on='species')

gene_stats = df_mf_all_genes_on_tree.groupby('gene', observed=True).agg(
    species_with_gene=('species', 'nunique'),
    sequence_count=('gene', 'size'),
    average_length=('length', 'mean'),
    std_length=('length', 'std'),
)

counts = df_mf_all_genes_on_tree.groupby(['species', 'gene'], observed=True).size().reset_index(name='copy_count')
copy_stats = counts.groupby('gene', observed=True)['copy_count'].agg(
    avg_copies_per_species='mean',
    std_copies_per_species='std'
)

gene_stats = gene_stats.join(copy_stats)

print(f"Number of species in both Fish Tree of Life and MitoFish after name remapping: {df_mf_all_genes_on_tree['species'].nunique()}")
gene_stats

Number of species in both Fish Tree of Life and MitoFish after name remapping: 11460


,species_with_gene,sequence_count,average_length,std_length,avg_copies_per_species,std_copies_per_species
gene,,,,,,
12S_rRNA,8013,32929,608.041483,324.352122,4.109447,8.625632
16S_rRNA,8711,39033,882.784259,558.030915,4.480886,9.582122
ATPase6,4742,17894,654.183805,80.512734,3.773513,11.365315
ATPase8,4598,16086,161.304986,23.489461,3.498478,10.648001
COXI,9865,141830,705.610026,273.353306,14.377091,27.309610
COXII,4139,10666,684.679074,44.233155,2.576951,8.156224
COXIII,4227,11165,731.707031,183.029114,2.641353,8.042603
Cytb,9085,109608,921.748723,278.539546,12.064722,36.778656
ND1,4498,13486,952.976865,113.680675,2.998221,10.939134


In [55]:
# 1. Basic counts per order
order_counts = df_mf_all_genes_on_tree.groupby('order', observed=True).agg(
    num_species=('species', 'nunique'),
    num_rows=('species', 'size')
)

# 2. Count unique genes per individual species (keeping the order context)
species_gene_counts = (
    df_mf_all_genes_on_tree
    .groupby(['order', 'species'], observed=True)['gene']
    .nunique()
    .reset_index(name='unique_gene_count')
)

# 3. Aggregate species-level data up to the order level
order_gene_stats = (
    species_gene_counts.groupby('order', observed=True)['unique_gene_count']
    .agg(
        avg_unique_genes_per_species='mean',
        std_unique_genes_per_species='std', # Added standard deviation
        frac_species_with_2plus_genes=lambda x: (x >= 2).mean()
    )
)

# 4. Join, sort, and display
final_order_stats = (
    order_counts.join(order_gene_stats)
    .sort_values(by='num_species', ascending=False)
)

final_order_stats.head(30)


,num_species,num_rows,avg_unique_genes_per_species,std_unique_genes_per_species,frac_species_with_2plus_genes
order,,,,,
Cypriniformes,1672,93180,9.354067,6.088043,0.923445
Perciformes,1252,50684,7.889776,5.848664,0.924920
Siluriformes,1136,24386,5.269366,4.801209,0.858275
Cichliformes,724,12202,4.723757,4.535514,0.835635
Cyprinodontiformes,580,12989,5.096552,4.228898,0.931034
Gobiiformes,511,24991,7.113503,5.429299,0.937378
Characiformes,481,11200,4.812890,4.259587,0.956341
Labriformes,338,8683,6.375740,5.243416,0.940828
Blenniiformes,265,4357,4.535849,4.796586,0.758491


## Export processed dataset

The final curated dataset is exported as a Parquet file.

The dataset includes:

- nucleotide sequences
- gene annotations
- species identifiers
- taxonomic lineage (order, family, genus, species)

This processed dataset will be used as input for the transformer model training pipeline.

In [56]:
path_out_dir = Path("output")
path_out_dir.mkdir(exist_ok=True)

path_processed_dataset = path_out_dir / "processed_dataset.parquet"

df_final = df_mf_all_genes_on_tree.astype({tax : 'category' for tax in ['order', 'family', 'genus', 'species'] })
df_final.to_parquet(path_processed_dataset)

df_final


,accession,gene,length,species,family,genus,order,sequence
0,AB000667,12S_rRNA,949,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCTCCACATCGGCCGAGGTCTATACTACGGCTCTTTTCTGTATAAA...
1,AB000667,16S_rRNA,86,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCTCCACATCGGCCGAGGTCTATACTACGGCTCTTTTCTGTATAAA...
2,AB000667,Cytb,856,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCTCCACATCGGCCGAGGTCTATACTACGGCTCTTTTCTGTATAAA...
3,AB000673,ND5,815,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CATTAGATTGTGATTCTAAAAACAGAGGTTGAAGCCCTCTTGCCCA...
4,AB000674,COXIII,233,Paralichthys olivaceus,Paralichthyidae,Paralichthys,Pleuronectiformes,CCCTTCACTATTGCAGATGGGGTTTATGGCGCCACATTCTTCGTTG...
...,...,...,...,...,...,...,...,...
473184,Z97552,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,GCAAACGACGCACTGGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
473185,Z97553,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,GCAAACGACGCACTGGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
473186,Z97554,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,GCAAACGACGCACTGGTTGATCTCCCAGCTCCTTCAAACATTTCCG...
473187,Z97555,Cytb,363,Tanganicodus irsacae,Cichlidae,Tanganicodus,Cichliformes,NNNNNNNNNNNNNNNGTTGATCTCCCAGCTCCTTCAAACATTTCCG...


In [57]:
path_processed_dataset_taxonomy = path_out_dir / "processed_dataset_taxonomy.csv"

df_taxonomy = df_final[['species', 'genus', 'family', 'order']].drop_duplicates()
df_taxonomy = df_taxonomy.sort_values(by='species', ignore_index=True)
df_taxonomy.to_csv(path_processed_dataset_taxonomy)

df_taxonomy

,species,genus,family,order
0,Abalistes stellaris,Abalistes,Balistidae,Tetraodontiformes
1,Abalistes stellatus,Abalistes,Balistidae,Tetraodontiformes
2,Abantennarius bermudensis,Abantennarius,Antennariidae,Lophiiformes
3,Abantennarius coccineus,Abantennarius,Antennariidae,Lophiiformes
4,Abantennarius nummifer,Abantennarius,Antennariidae,Lophiiformes
...,...,...,...,...
11455,Zosterisessor ophiocephalus,Zosterisessor,Gobiidae,Gobiiformes
11456,Zu cristatus,Zu,Trachipteridae,Lampriformes
11457,Zu elongatus,Zu,Trachipteridae,Lampriformes
11458,Zungaro jahu,Zungaro,Pimelodidae,Siluriformes
